In [3]:
import cv2
import numpy as np

img = cv2.imread("s58Hl.jpg")

if img is None:
    print("Error: Image not found.")
    exit()


debug_learning_img = img.copy()

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
# --- Step 1: Raw Color Detection ---
lower_purple = np.array([125, 50, 40])
upper_purple = np.array([165, 255, 255])
raw_mask = cv2.inRange(hsv, lower_purple, upper_purple)

# SHOW STEP 1
cv2.imshow("Step 1: Raw Color Mask", raw_mask)



contours, _ = cv2.findContours(raw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
single_cell_areas = []
for c in contours:
    area = cv2.contourArea(c)
    x, y, w, h = cv2.boundingRect(c)
    ar = w / h
    
    # Logic to find "Perfect" single cells
    if area > 60 and 0.8 < ar < 1.2 : 
        single_cell_areas.append(area)
        
        # VISUALIZE: Draw BLUE boxes around the cells used for learning
        cv2.rectangle(debug_learning_img, (x, y), (x+w, y+h), (255, 0, 0), 2)

if not single_cell_areas:
    print("Could not find any single cells to learn from.")
    exit()

# SHOW STEP 3
cv2.imshow("Step 3: Learning Standard (Blue)", debug_learning_img)

# Calculate Standard Size
reference_size = np.median(single_cell_areas)
print(f"Learned Single Cell Size: {reference_size:.2f}")

# --- Step 4: Final Counting ---
total_cells = 0

for c in contours:
    area = cv2.contourArea(c)
    
    if area < 70: continue

    x, y, w, h = cv2.boundingRect(c)

    # Calculate overlaps
    num_cells_in_blob = int(round(area / reference_size))
    if num_cells_in_blob < 1:
        num_cells_in_blob = 1

    total_cells += num_cells_in_blob
    # Color coding: Green for single cells, Red for clumps
    if num_cells_in_blob == 1:
        color = (0, 255, 0) 
    else:
        color = (0, 0, 255) 
    cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)
    cv2.putText(img, str(num_cells_in_blob), (x, y-5), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

print(f"Total estimated WBC nuclei: {total_cells}")

# SHOW STEP 4 (Final)
cv2.imshow("Step 4: Final Counting (Green=1, Red=Clump)", img)

cv2.waitKey(0)
cv2.destroyAllWindows()

Learned Single Cell Size: 472.75
Total estimated WBC nuclei: 235
